# 04 時間、地點、人 — 筆記本

Line list（造冊）已經乾淨了（第 03 章）。現在 Maya 要用每一位現場流行病學家都會從這裡開始的方式，描述這場疫情：人們**什麼時候**發病、**住在哪裡**、**是誰**？你在這裡看到的每個數字，都是從合成的 Santa Luz 資料計算出來的。Santa Luz 市、颱風 Amihan 和所有病例都是為了教學而虛構的。

你會做出：病例分類表、標示颱風的流行曲線、依地帶與 barangay 計算的每 10,000 人侵襲率、年齡－性別圖，以及致死率。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 步驟 1 · 載入乾淨的 line list 和分母

兩個檔案：**病例**（`santa_luz_linelist.csv`，每個病例一列）和各 barangay 的**人口**（`santa_luz_barangays.csv`）。我們告訴 pandas 哪些欄位是日期，這樣它才會把它們讀成真正的日期，而不是文字。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
# BEGIN-CJK (中文字型設定：避免標籤顯示為方框)
import pathlib
import matplotlib.font_manager as fm
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass
_discovered = []
for _entry in fm.fontManager.ttflist:
    if any(_kw in _entry.name.lower() for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = ["Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans TC",
              "Microsoft JhengHei", "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
              "Heiti TC", "DejaVu Sans"]
_cjk_fonts = _discovered + [n for n in _preferred if n not in _discovered]
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts
# END-CJK

DATE_COLUMNS = ["date_onset", "date_consult", "date_report", "date_lab_confirmed"]
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=DATE_COLUMNS)
barangays = pd.read_csv("data/synthetic/ph/santa_luz_barangays.csv")

LANDFALL = pd.Timestamp("2026-07-18")  # 颱風 Amihan
BANDS = ["riverside", "coastal", "upland"]

print(f"{len(ll)} 個病例 | {len(barangays)} 個 barangay | {barangays['population'].sum():,} 位居民")
ll["disease"].value_counts()

## 步驟 2 · 病例分類

每一列在 9 月 30 日資料凍結時，都已依病例定義分類：**疑似（suspect）**（符合臨床描述）、**可能（probable）**（再加上快篩陽性）、**確診（confirmed）**（再加上實驗室檢驗陽性）。交叉表會計算每一種組合；`margins=True` 會加上總計。

In [ ]:
classes = pd.crosstab(ll["disease"], ll["case_class"], margins=True, margins_name="Total")
classes[["confirmed", "probable", "suspect", "Total"]]

## 步驟 3 · 流行曲線（時間）

> **問問 Tala：**「我的 DataFrame `lepto` 有一個 datetime 欄位 `date_onset`（虛構的資料列：`2026-07-24`、`2026-07-24`、`2026-07-27`）。請畫一張流行曲線：從 2026 年 7 月 15 日到 8 月 31 日，每天一根長條，**包含病例數為零的日子**，長條之間沒有空隙。在 7 月 18 日畫一條紅色虛線，標示 'Typhoon Amihan landfall'，並把洪水的日子 7 月 18–20 日，以及清理期 7 月 21 日至 8 月 2 日用陰影標出。」

重要的技巧是 `reindex(..., fill_value=0)`：`value_counts()` 只列出*有*病例的日子，所以我們把這些計數放到一個完整的日曆上。

In [ ]:
lepto = ll[ll["disease"] == "leptospirosis"]
days = pd.date_range("2026-07-15", "2026-08-31", freq="D")
daily = lepto["date_onset"].value_counts().reindex(days, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axvspan(pd.Timestamp("2026-07-17 12:00"), pd.Timestamp("2026-07-20 12:00"),
           color="#4FA3D9", alpha=0.30, label="洪水（7 月 18–20 日）")
ax.axvspan(pd.Timestamp("2026-07-20 12:00"), pd.Timestamp("2026-08-02 12:00"),
           color="#FCD116", alpha=0.25, label="清理期（7 月 21 日至約 8 月 2 日）")
ax.bar(daily.index, daily.values, width=1.0, color="#0038A8", edgecolor="white", linewidth=0.6)
ax.axvline(LANDFALL, color="#CE1126", linestyle="--", linewidth=2,
           label="颱風 Amihan 登陸（7 月 18 日）")
ax.set_title("Santa Luz 市鉤端螺旋體病病例，依發病日（合成資料）")
ax.set_xlabel("發病日（2026 年）")
ax.set_ylabel("病例數")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**檢查一下。** 長條加起來必須等於每一個**有**發病日的鉤端螺旋體病病例（有 2 例沒有，見第 03 章規則 C4）。每週總數必須和疫情報告相符。然後找出高峰日。

In [ ]:
missing_onset = lepto["date_onset"].isna().sum()
assert daily.sum() == len(lepto) - missing_onset
print(f"{daily.sum()} 根長條 + {missing_onset} 例沒有發病日 = {len(lepto)} 個病例")

weekly = lepto["date_onset"].dt.isocalendar().week.value_counts().sort_index()
weekly.index = weekly.index.astype(int)
print("依 ISO 發病週的病例數：", weekly.to_dict())
assert weekly.to_dict() == {30: 35, 31: 108, 32: 68, 33: 15, 34: 2}
print("高峰日：", daily.idxmax().date(), "，共", daily.max(), "例")

## 步驟 4 · 這個形狀說了什麼？

像偵探一樣解讀這條曲線：

- **第一個發病日是 7 月 22 日**，登陸後四天（強化版 line list 從 7 月 20 日開始收集發病日）。人們需要時間才會生病：這就是**潛伏期（incubation period）**，從感染到第一個症狀的時間。
- **一個波**：快速上升，維持高點大約兩週（7 月 24 日至 8 月 6 日），然後在 8 月中旬消退。沒有第二、第三或第四個波。
- 鉤端螺旋體病通常不會人傳人，而且這個波與洪水和清理期吻合，當時人們持續涉入可能受汙染的水。

這是一個人們在幾天之內暴露其中的**共同暴露源**的特徵，也就是**持續性共同暴露源（continuing common source）**；不是單一時間點（*單一暴露源* 會給出更尖銳的單一高峰），也不是*傳播型*疫情（連續的波，每個波相隔一個潛伏期）。從登陸日起算天數，讓這個推理更具體。

作為比較，我們也依週畫出**登革熱**曲線：登革熱由在積水中孳生的*斑蚊（Aedes）*傳播，所以它上升得比較晚，而且會連續攀升好幾週。

In [ ]:
days_after = (lepto["date_onset"] - LANDFALL).dt.days.dropna().astype(int)
print("登陸日到發病日的天數：最早", days_after.min(), "| 中位數", int(days_after.median()),
      "| 最晚", days_after.max())

dengue = ll[ll["disease"] == "dengue"]
dengue_weeks = dengue["date_onset"].dt.isocalendar().week.value_counts().reindex(range(30, 40), fill_value=0)
print("依 ISO 發病週的登革熱病例數：", dengue_weeks.to_dict())

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"W{w}" for w in dengue_weeks.index], dengue_weeks.values, width=0.95, color="#00737D")
ax.set_title("2026 年登革熱病例，依發病週（合成資料；W39 尚未完整）")
ax.set_xlabel("ISO 發病週")
ax.set_ylabel("病例數")
plt.tight_layout()
plt.show()

## 步驟 5 · Tala 搞錯了：沒有人口數的「侵襲率」

> **問問 Tala：**「依 `band` 計算登革熱侵襲率。」

Tala 的答案可以執行，看起來也很整齊：

In [ ]:
# Tala 的第一個答案：各地帶占「病例」的比例，不是侵襲率
wrong = (dengue["band"].value_counts(normalize=True) * 100).round(1)
print("Tala 的「侵襲率（%）」：")
print(wrong)
riverside_share = (lepto["band"] == "riverside").mean() * 100
print(f"鉤端螺旋體病犯了同樣的錯誤：{riverside_share:.0f}% 的病例是河岸居民")

這些數字加起來是 100 %。這就是線索：它們說的是**病例在哪裡**，不是**住在那裡有多危險**。河岸的登革熱病例最多，只是因為那裡的人最多（186,000 人中有 96,064 人）。**侵襲率**是

$$\text{侵襲率} = \frac{\text{一個群體中的新發病例數}}{\text{該群體中的危險族群人數}}$$

對一座城市，我們以**每 10,000 居民**表示。分母來自 barangay 檔案，不是來自 line list。

In [ ]:
population = barangays.groupby("band")["population"].sum()
cases = pd.crosstab(ll["band"], ll["disease"])

attack = cases.div(population, axis=0) * 10_000
attack.loc["city"] = cases.sum() / population.sum() * 10_000
attack = attack.loc[BANDS + ["city"], ["leptospirosis", "dengue"]].round(1)
print("每 10,000 居民的侵襲率")
attack

用手算其中一格來**檢查**，並比較兩個答案。對登革熱來說，Tala 的版本把河岸排在第一；真正的侵襲率說，風險最高的是**沿海**地帶。

In [ ]:
riverside_lepto = (ll["disease"].eq("leptospirosis") & ll["band"].eq("riverside")).sum()
print(f"河岸鉤端螺旋體病：{riverside_lepto} / {population['riverside']:,} x 10,000 =",
      round(riverside_lepto / population["riverside"] * 10_000, 1))
assert attack.loc["riverside", "leptospirosis"] == 19.7
assert attack.loc["coastal", "dengue"] == 29.6
assert attack["dengue"].drop("city").idxmax() == "coastal" != wrong.idxmax()

## 步驟 6 · 地點：依 barangay 的侵襲率

現在對 24 個 barangay 各自套用同樣的想法。我們**從 barangay 表開始**，再把病例計數併進去，這樣沒有病例的 barangay（山區的 Bukid）仍然會出現，侵襲率是 0，而不是缺漏。

In [ ]:
counts = ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
place = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts).fillna(0)
place[["leptospirosis", "dengue"]] = place[["leptospirosis", "dengue"]].astype(int)
for disease in ["leptospirosis", "dengue"]:
    place[f"ar_{disease}"] = (place[disease] / place["population"] * 10_000).round(1)

print(len(place), "個 barangay；", (place["leptospirosis"] + place["dengue"] == 0).sum(), "個沒有病例")
top = place.sort_values("ar_leptospirosis", ascending=False)
top[["barangay", "band", "leptospirosis", "population", "ar_leptospirosis"]].head(4)

依侵襲率排序、依地帶上色的長條圖，一眼就能看出模式（第 08 章會把它畫成地圖）。

In [ ]:
BAND_COLOURS = {"riverside": "#0038A8", "coastal": "#00737D", "upland": "#2F7D32"}
DISEASE_ZH = {"leptospirosis": "鉤端螺旋體病", "dengue": "登革熱"}
fig, axes = plt.subplots(1, 2, figsize=(11, 6.5), sharey=False)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    data = place.sort_values(f"ar_{disease}")
    ax.barh(data["barangay"], data[f"ar_{disease}"], color=data["band"].map(BAND_COLOURS))
    ax.set_title(DISEASE_ZH[disease])
    ax.set_xlabel("每 10,000 居民的病例數")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in BAND_COLOURS.values()]
fig.legend(handles, ["河岸", "沿海", "山區"], loc="lower center", ncol=3)
fig.suptitle("Santa Luz 市各 barangay 的侵襲率（合成資料）")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

print("登革熱，最高的兩個：", place.nlargest(2, "ar_dengue")[["barangay", "ar_dengue"]].values.tolist())

## 步驟 7 · 人：年齡、性別與死亡

誰生病了？為每種疾病畫一張並排的**年齡－性別圖**（男性在左，女性在右），再加上年齡中位數和男性所占比例。有七例登革熱病例沒有年齡，所以我們另外計算它們，而不是讓它們憑空消失。

然後是**致死率（case-fatality rate, CFR）**：在成為病例的人當中，有多少比例死亡？這裡的分母*就是*病例數，這是定義，與侵襲率的錯誤正好相反。

In [ ]:
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]
DISEASE_ZH = {"leptospirosis": "鉤端螺旋體病", "dengue": "登革熱"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, disease in zip(axes, ["leptospirosis", "dengue"]):
    d = ll[ll["disease"] == disease]
    table = pd.crosstab(d["age_group"], d["sex"]).reindex(AGE_GROUPS, fill_value=0)
    ax.barh(AGE_GROUPS, -table["M"], color="#0038A8", label="男")
    ax.barh(AGE_GROUPS, table["F"], color="#CE1126", label="女")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{DISEASE_ZH[disease]}（n = {len(d)}）")
    ax.set_xlabel("病例數（男性在左，女性在右）")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{abs(int(x))}")
axes[0].set_ylabel("年齡組（歲）")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

for disease in ["leptospirosis", "dengue"]:
    d = ll[ll["disease"] == disease]
    male = (d["sex"] == "M").sum()
    print(f"{disease:<14} 年齡中位數 {d['age'].median():.0f} | 男性 {male}（{male / len(d):.0%}）"
          f" | 年齡缺漏 {d['age'].isna().sum()}")

# 致死率：死亡數 / 「病例數」（這裡的病例數「就是」對的分母）
cfr = ll.groupby("disease").agg(cases=("case_id", "size"),
                                deaths=("outcome", lambda s: (s == "died").sum()),
                                hospitalised=("hospitalised", "sum"))
cfr["cfr_pct"] = (cfr["deaths"] / cfr["cases"] * 100).round(1)
cfr["hospitalised_pct"] = (cfr["hospitalised"] / cfr["cases"] * 100).round(1)
assert cfr.loc["leptospirosis", "deaths"] == 20 and cfr.loc["leptospirosis", "cfr_pct"] == 8.7
cfr

## 換你試試

1. **登革熱的每日流行曲線。** 複製鉤端螺旋體病曲線的儲存格，畫出 7 月 15 日到 9 月 30 日的登革熱發病日曲線。它看起來像是來自單一來源的一個波嗎？*提示：改掉篩選條件和 `date_range`；省略清理期的陰影。*
2. **依性別的侵襲率。** barangay 檔案沒有性別的分項，所以假設一半人口是男性。計算每 10,000 名男性與每 10,000 名女性的鉤端螺旋體病病例數。*提示：`lepto["sex"].value_counts() / (186_000 / 2) * 10_000`，並在你的說明中清楚寫明，分母是一個假設。*
3. **依年齡組的致死率。** 鉤端螺旋體病在年長的病例中比較致命嗎？*提示：`lepto.groupby("age_group")["outcome"].apply(lambda s: (s == "died").mean())`，並同時印出每組的病例數，因為從 9 個病例算出來的致死率並不可靠。*